# 03 — Fermat's Nightmare: What Falls and What Survives

**The Zero Tree / Telperion — Notebook 3 of 4**

---

**Fermat's Last Theorem** says: `x^n + y^n = z^n` has no positive integer solutions for `n ≥ 3`.

In the CD tower this is **not** a separate fact — it is a consequence of the same  
zero-divisor structure that appears at k=4 (𝕊, dim=16):

```
Hurwitz 1898:  Only ℝ(1), ℂ(2), ℍ(4), 𝕆(8) are normed division algebras.
At dim=16 (𝕊): ZD first appears.  |ab|=|a||b| FAILS.

Fermat n=2 (Pythagorean): ℂ-norm holds → solutions ABUNDANT.
Fermat n≥3: ZD at dim=16 kills multiplicativity → solutions EXTINCT.

The SAME algebraic failure — two mathematical languages for one identity.
```

**What falls at k=4:**
Every composite `n = a × b` can be written as a product of factors.  
At the sedenion level (k=4), those factors can be arranged as a ZD pair:  
two non-zero elements whose product is zero.  
The composite's norm fails. It falls off the tree.

**What survives:**
A prime p has NO non-trivial factorization.  
No ZD pair can express it.  
It reaches T_256 intact.

**The fractal boundary:**
The boundary at k=4 between survivors (primes) and fallen (composites)  
has fractal structure because the prime distribution is fractal:  
- The density of primes below x is π(x) ~ x/ln(x)  
- Oscillations in π(x) are governed by Riemann zeros  
- Those same zeros are the lens of the Zero Tree (wiki #72)  
- The boundary IS the spectral structure of the tree, frozen at k=4

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'engine'))
from telperion_engine import (
    prime_sieve, fermat_survival_table, fractal_boundary_data,
    prime_density_at_level, FIRST_ZD_LEVEL, NIEMEIER_GAP,
)

# Pull Fermat verification from the Monster engine
sys.path.insert(0, os.path.join('..', '..', 'FermatMonster', 'engine'))
from fermat_monster_engine import cd_tower_check, fermat_niemeier_bridge

# CD tower norm check
cd = cd_tower_check()
print('Cayley-Dickson norm check:')
for key, v in cd.items():
    if isinstance(v, dict) and 'dim' in v:
        print(f'  {v["algebra"]:4s} dim={v["dim"]:3d}: norm_holds={v["norm_holds"]}  '
              f'division_algebra={v["is_division_algebra"]}  [{v["fermat_n"]}]')
    elif isinstance(v, str):
        print(f'  Hurwitz: {v[:90]}')

In [ ]:
# Fermat solution counts (from FermatMonster engine)
br = fermat_niemeier_bridge()
b4 = br['B4']
print('Fermat solution counts:')
print(f'  n=2 (Pythagorean), c≤200:  {b4["n2_pythagorean_solutions_c_le_200"]} solutions — {b4["n2_status"]}')
print(f'  n=3, c≤100:                {b4["n3_solutions_c_le_100"]} solutions — {b4["n3_status"]}')
print(f'  n=4, c≤100:                {b4["n4_solutions_c_le_100"]} solutions — {b4["n4_status"]}')
print(f'  n=5, c≤50:                 {b4["n5_solutions_c_le_50"]} solutions — {b4["n5_status"]}')
print()
print(b4['bridge_statement'])

In [ ]:
# Survival table for n ≤ 100
surv = fermat_survival_table(100)
print(f'N={surv["N"]}: {surv["n_primes"]} survivors (primes), {surv["n_composites"]} fallen (composites)')
print()
print('Level-by-level survival stats:')
print(f'  {"k":>3}  {"is_ZD":>6}  {"surviving":>10}  {"fallen":>8}')
for k, st in surv['level_stats'].items():
    print(f'  k={k}  {str(st["is_zd"]):>6}  {st["n_surviving"]:10d}  {st["n_fallen"]:8d}')

## The fractal boundary at k=4

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

N      = 1000
primes = prime_sieve(N)
fb     = fractal_boundary_data(primes)

# Polar plot of prime density by N-shape at k=4 (ZD equator = the boundary)
k_plot = 4  # the equator — where composites fall
lv_data = fb['level_boundary'][k_plot]

ns_vals  = list(range(16))
radii    = [lv_data['nshape_density'][ns]['radius']   for ns in ns_vals]
phis_rad = [np.radians(ns * 22.5)                      for ns in ns_vals]
is_gap   = [ns in NIEMEIER_GAP                         for ns in ns_vals]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': 'polar'})

for ax, k_val, title in [
    (axes[0], 0, 'k=0 (ℝ leaf level — all paths present, full circle)'),
    (axes[1], 4, 'k=4 (𝕊 equator — composites fallen, fractal boundary)'),
]:
    lv = fb['level_boundary'][k_val]
    r_list  = [lv['nshape_density'][ns]['radius'] for ns in ns_vals]
    colors  = ['silver' if ns in NIEMEIER_GAP else
               ('#334466' if ns in {1,3,5,7,9,11,13,15} else '#888888')
               for ns in ns_vals]
    for phi, r, col in zip(phis_rad, r_list, colors):
        ax.bar(phi, r, width=np.radians(20), color=col, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(title, pad=15, fontsize=9)
    ax.set_rticks([])
    ax.set_xticks(phis_rad[::2])
    ax.set_xticklabels([f'e{ns}' for ns in ns_vals[::2]], fontsize=7)

gap_patch = mpatches.Patch(color='silver', label='Monster gap {e₁,e₁₁,e₁₅}')
prime_patch = mpatches.Patch(color='#334466', label='Other prime-sector N-shapes')
fig.legend(handles=[gap_patch, prime_patch], loc='lower center', ncol=2)
plt.suptitle('Fractal boundary: prime density by N-shape at k=0 vs k=4', y=1.02)
plt.tight_layout()
plt.savefig('03_fractal_boundary.png', dpi=150)
plt.show()

In [ ]:
# V(n) modulation of the boundary across levels
fig, ax = plt.subplots(figsize=(12, 5))

from fixed_point import v_nball

for ns in range(16):
    if ns not in {1, 3, 5, 7, 9, 11, 13, 15}:  # skip even (only p=2)
        continue
    radii_by_k = []
    for k in range(9):
        r = fb['level_boundary'][k]['nshape_density'][ns]['radius']
        radii_by_k.append(r)
    col = 'silver' if ns in NIEMEIER_GAP else '#334466'
    lw  = 2.0       if ns in NIEMEIER_GAP else 0.8
    ax.plot(range(9), radii_by_k, '-o', color=col, linewidth=lw,
            markersize=5, label=f'e{ns}' if ns in NIEMEIER_GAP else None, alpha=0.7)

ax.axvline(FIRST_ZD_LEVEL, color='red', linestyle='--', linewidth=1.5, label='k=4: first ZD (composites fall here)')
ax.set_xticks(range(9))
ax.set_xticklabels([f'k={k}\n{["ℝ","ℂ","ℍ","𝕆","𝕊","t32","t64","t128","T256"][k]}' for k in range(9)])
ax.set_xlabel('CD level k')
ax.set_ylabel('Boundary radius (prime density × V-factor)')
ax.set_title('Fractal boundary radius by N-shape across CD levels\n(silver = Monster gap, blue = Niemeier)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('03_boundary_across_levels.png', dpi=150)
plt.show()

## Why the boundary is fractal

The prime counting function π(x) has the **explicit formula**:

```
π(x) = li(x) - Σ_ρ li(x^ρ) - log(2) + ∫_x^∞ dt/(t(t²-1)log(t))
```

where the sum is over Riemann zeros `ρ = ½ + iγ`.

Each zero `γ` contributes an oscillation of period `log(x) / γ`.  
More zeros → more oscillations → finer fractal detail.

**The zeros ARE the spectral nodes of the Zero Tree.**  
The boundary inherits the tree's own structure.
It is not decorative — it is the Riemann spectrum made visible.

In [ ]:
# π(x) vs li(x) — the oscillation is the fractal boundary
import math
from scipy.special import expi  # logarithmic integral li(x)

x_vals = list(range(3, 1001))
primes_set = set(prime_sieve(1000))
pi_x   = [sum(1 for p in primes_set if p <= x) for x in x_vals]
li_x   = [expi(math.log(x)) for x in x_vals]   # li(x) = Ei(ln x)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(x_vals, pi_x, color='#334466', linewidth=1, label='π(x)')
axes[0].plot(x_vals, li_x, color='red', linewidth=1, linestyle='--', label='li(x)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('Count')
axes[0].set_title('π(x) vs li(x) — the gap is the oscillation sum Σ li(x^ρ)')
axes[0].legend()

# Difference: the fractal oscillation
diff = [li - pi for li, pi in zip(li_x, pi_x)]
axes[1].plot(x_vals, diff, color='#884422', linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel('li(x) − π(x)')
axes[1].set_title('li(x) − π(x): the fractal boundary oscillation\n(driven by Riemann zeros = lens of the Zero Tree)')

plt.tight_layout()
plt.savefig('03_pi_fractal.png', dpi=150)
plt.show()